In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

/mnt/d/Projects/MIRx/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
file = r'/content/Preprocessed Medicine Dataset.csv'
data = pd.read_csv(file)
data = pd.DataFrame(data)
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data['Chemical Class'] = data['Chemical Class'].fillna('unknown')
data['Action Class'] = data['Action Class'].fillna('unknown')
data['Therapeutic Class'] = data['Therapeutic Class'].fillna('unknown')
data['Substitutes'] = data['Substitutes'].fillna('unknown')

Confidence score is calculated because there is some unknown data. If there is more unknown data, the score is high and vice-versa. The higher the score, the lesser the reliability.

In [ ]:
def compute_confidence(row):
    score = 0
    if row['Chemical Class'] != 'unknown':
        score += 1
    if row['Action Class'] != 'unknown':
        score += 1
    if row['Therapeutic Class'] != 'unknown':
        score += 1
    if row['Substitutes'] != 'unknown':
        score += 1

    return score

data['Confidence Score'] = data.apply(compute_confidence, axis = 1)
data.head()

Creating documents

In [ ]:
def create_document(row):
    document = f"Name: {row['name']}\n"

    if row['Chemical Class'] != 'unknown':
        document += f"Chemical Class: {row['Chemical Class']}\n"
    if row['Action Class'] != 'unknown':
        document += f"Action Class: {row['Action Class']}\n"
    if row['Therapeutic Class'] != 'unknown':
        document += f"Therapeutic Class: {row['Therapeutic Class']}\n"
    if row['Substitutes'] != 'unknown':
        document += f"Substitutes: {row['Substitutes']}"

    document += f"""
        Habit Forming: {row['Habit Forming']}\n
        Side Effects: {row['Side Effects']}\n
        Uses: {row['Uses']}\n
        Confidence Score: {row['Confidence Score']}\n
        """

    return document

data['Docs'] = data.apply(create_document, axis = 1)
data.head()


Model and Indexing

In [ ]:
model = SentenceTransformer('all-miniLM-L6-v2', device = 'cuda')

In [ ]:
embeddings = model.encode(
    data['Docs'].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(embeddings.shape)

In [ ]:
np.save("embeddings.npy", embeddings.cpu().numpy())

Retrieval